# Unit I — AI in Pharmacokinetics & Dose Optimization
**Pharmakaksha · Learn · Grow · Achieve** — BP703T AI in Clinical Applications (B.Pharm Sem VII, PCI NEP 2020)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajreddys/learn-pharmakaksha-site/blob/main/content/BP703T%20AI%20in%20Clinical%20Applications%20%28Theory%29/BP703T_Unit1_AI_in_Pharmacokinetics.ipynb)

Every example from the Unit I notes, slides and video, in order, followed by lab exercises and solutions. Data files: `pk_profile.csv`, `dose_cohort.csv` (fictional teaching data).

## Using learn.pharmakaksha.com
- **First run is slow.** Python starts inside your browser; the first cell can take about 10 seconds. Later cells are fast.
- **Kernel indicator.** The circle at the top right is filled while Python is busy. Wait for it to clear.
- **Stuck?** Use *Kernel → Restart Kernel*, then run the cells again from the top.
- **Saving.** Your changes are saved in this browser only. Use *File → Download* to keep a copy.
- **Start over.** Delete your copy from the file list on the left and reopen the notebook from the course folder.

In [ ]:
import os
import numpy as np
import pandas as pd
BASE = "https://raw.githubusercontent.com/ajreddys/learn-pharmakaksha-site/main/content/BP703T%20AI%20in%20Clinical%20Applications%20%28Theory%29/"

def load(name):
    """Read a practice CSV: local copy first, then the course's GitHub folder."""
    if os.path.exists(name):
        return pd.read_csv(name)
    try:
        return pd.read_csv(BASE + name)
    except Exception:
        from pyodide.http import open_url   # inside JupyterLite
        return pd.read_csv(open_url(BASE + name))

## 1.1 PK parameters from one profile
One patient took 500 mg of Drug V by mouth. We read Cmax, Tmax and AUC from 11 blood samples.

In [ ]:
p = load("pk_profile.csv")
p

In [ ]:
i = p["conc_mg_L"].idxmax()
cmax, tmax = p.loc[i, "conc_mg_L"], p.loc[i, "time_h"]
auc_0_24 = np.trapezoid(p["conc_mg_L"], p["time_h"])   # older NumPy: np.trapz
print("Cmax =", cmax, "mg/L")
print("Tmax =", tmax, "h")
print("AUC0-24 =", round(auc_0_24, 1), "mg·h/L")

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 5))
plt.fill_between(p["time_h"], p["conc_mg_L"], alpha=0.2, color="#1F8A70", label="AUC")
plt.plot(p["time_h"], p["conc_mg_L"], "o-", color="#12303A")
plt.annotate(f"Cmax {cmax} mg/L at {tmax} h", xy=(tmax, cmax), xytext=(5, 8), arrowprops=dict(arrowstyle="->"))
plt.xlabel("Time (h)"); plt.ylabel("Concentration (mg/L)")
plt.title("Concentration–time profile"); plt.legend(); plt.show()

## 1.2 Linear regression on ln C: elimination rate and half-life
After absorption, ln C falls in a straight line. Slope = −k, t½ = 0.693 / k.

In [ ]:
from sklearn.linear_model import LinearRegression
el = p[p["time_h"] >= 6]
log_model = LinearRegression().fit(el[["time_h"]], np.log(el["conc_mg_L"]))
k = -log_model.coef_[0]
t_half = np.log(2) / k
print("k =", round(k, 3), "per h")
print("t½ =", round(t_half, 2), "h")
print("R² on ln C =", round(log_model.score(el[["time_h"]], np.log(el["conc_mg_L"])), 3))

In [ ]:
c_last = p["conc_mg_L"].iloc[-1]
auc_inf = auc_0_24 + c_last / k
cl_f = 500 / auc_inf
print("AUC0-inf =", round(auc_inf, 1), "mg·h/L")
print("CL/F =", round(cl_f, 2), "L/h")

## 1.3 Multiple regression for clearance
150 patients on Drug V. Model A uses raw covariates; Model B uses creatinine clearance (Cockcroft–Gault).

In [ ]:
c = load("dose_cohort.csv")
c["female"] = (c["sex"] == "F").astype(int)
c.head()

In [ ]:
c.describe().round(2)

In [ ]:
from sklearn.model_selection import train_test_split
y = c["clearance_L_h"]
X_a = c[["age", "weight_kg", "serum_creatinine", "female"]]
X_tr, X_te, y_tr, y_te = train_test_split(X_a, y, test_size=0.25, random_state=42)
model_a = LinearRegression().fit(X_tr, y_tr)
print("Intercept:", round(model_a.intercept_, 2))
for name, b in zip(X_a.columns, model_a.coef_):
    print(f"{name:18s} {b:+.3f}")

Cockcroft–Gault: CrCl = (140 − age) × weight / (72 × SCr), × 0.85 if female. It is already a column in the file; here we recompute it to check.

In [ ]:
def crcl(age, wt, scr, female):
    v = (140 - age) * wt / (72 * scr)
    return v * 0.85 if female else v

check = crcl(c["age"], c["weight_kg"], c["serum_creatinine"], False) * np.where(c["female"] == 1, 0.85, 1)
print("Matches file:", np.allclose(check.round(1), c["crcl_mL_min"], atol=0.1))

In [ ]:
X_b = c[["crcl_mL_min"]]
Xb_tr, Xb_te, _, _ = train_test_split(X_b, y, test_size=0.25, random_state=42)
model_b = LinearRegression().fit(Xb_tr, y_tr)
print(f"CL = {model_b.intercept_:.2f} + {model_b.coef_[0]:.4f} × CrCl")

## 1.4 Coefficients in clinical context
Read each Model A coefficient as: *change in clearance (L/h) for one unit more of this covariate, others held fixed.*

In [ ]:
for name, b, step in zip(X_a.columns, model_a.coef_, [10, 10, 0.5, 1]):
    print(f"{name}: +{step} unit(s) → clearance changes by {b*step:+.2f} L/h")

## 1.5 RMSE and R² on the test set

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score
def report(name, y_true, y_pred):
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    print(f"{name:32s} RMSE {rmse:.2f} L/h   R² {r2_score(y_true, y_pred):.3f}")

report("Baseline (training mean)", y_te, np.full(len(y_te), y_tr.mean()))
report("A: age + weight + SCr + sex", y_te, model_a.predict(X_te))
report("B: CrCl", y_te, model_b.predict(Xb_te))

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_te, model_a.predict(X_te), color="#E07A2E", label="Model A")
plt.scatter(y_te, model_b.predict(Xb_te), color="#1F8A70", label="Model B (CrCl)")
plt.plot([0, 14], [0, 14], "--", color="grey")
plt.xlabel("Observed clearance (L/h)"); plt.ylabel("Predicted (L/h)")
plt.title("Test set: predicted vs observed"); plt.legend(); plt.show()

## 1.6 Limits of linear models
A straight line through raw concentrations goes below zero. The same points on the ln scale do not.

In [ ]:
raw_model = LinearRegression().fit(el[["time_h"]], el["conc_mg_L"])
print("R² on raw C =", round(raw_model.score(el[["time_h"]], el["conc_mg_L"]), 3))
future = pd.DataFrame({"time_h": [24, 30]})
print("Straight line predicts:", raw_model.predict(future).round(2), "mg/L")
print("Log model predicts:  ", np.exp(log_model.predict(future)).round(2), "mg/L")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 4.5))
axs[0].scatter(c["serum_creatinine"], c["clearance_L_h"], color="#E07A2E")
axs[0].set_xlabel("Serum creatinine (mg/dL)"); axs[0].set_ylabel("Clearance (L/h)"); axs[0].set_title("Curved")
axs[1].scatter(c["crcl_mL_min"], c["clearance_L_h"], color="#1F8A70")
axs[1].set_xlabel("CrCl (mL/min)"); axs[1].set_title("Straight")
plt.tight_layout(); plt.show()

## Worked example: a starting dose
Target AUC0–24 = 400 mg·h/L. Daily dose = 400 × predicted clearance.

In [ ]:
patients = pd.DataFrame({
    "name": ["Mrs. R", "Mr. K"], "age": [72, 30], "weight_kg": [60, 75],
    "scr": [1.6, 0.9], "female": [True, False]})
patients["crcl_mL_min"] = [round(crcl(r.age, r.weight_kg, r.scr, r.female), 1) for r in patients.itertuples()]
patients["pred_CL_L_h"] = model_b.predict(patients[["crcl_mL_min"]]).round(2)
patients["dose_mg_day"] = (400 * patients["pred_CL_L_h"]).round(-1)
patients

A real pharmacist would round to an available strength, check the label's renal dosing table, and confirm with a drug level (TDM).

---
# Lab exercises
1. Compute Cmax, Tmax, AUC0–24, k, t½, AUC0–∞ and CL/F from `pk_profile.csv`. *(Done above — try it yourself first.)*
2. Refit k using only the last three points (8, 12, 24 h). How much does t½ change?
3. Fit clearance on CrCl **and** weight. Does weight still help once CrCl is in? Compare test RMSE.
4. Predict the daily dose for a 55-year-old man, 82 kg, SCr 2.2 mg/dL, target AUC 400 mg·h/L.

## Solutions

In [ ]:
# Lab 2
last3 = p[p["time_h"] >= 8]
m3 = LinearRegression().fit(last3[["time_h"]], np.log(last3["conc_mg_L"]))
k3 = -m3.coef_[0]
print("k =", round(k3, 3), " t½ =", round(np.log(2) / k3, 2), "h  (was", round(t_half, 2), "h)")

In [ ]:
# Lab 3
X_c = c[["crcl_mL_min", "weight_kg"]]
Xc_tr, Xc_te, _, _ = train_test_split(X_c, y, test_size=0.25, random_state=42)
model_c = LinearRegression().fit(Xc_tr, y_tr)
print("Coefficients:", dict(zip(X_c.columns, model_c.coef_.round(4))))
report("B: CrCl", y_te, model_b.predict(Xb_te))
report("C: CrCl + weight", y_te, model_c.predict(Xc_te))

In [ ]:
# Lab 4
cr = crcl(55, 82, 2.2, False)
cl = model_b.predict(pd.DataFrame({"crcl_mL_min": [cr]}))[0]
print("CrCl =", round(cr, 1), "mL/min;  CL =", round(cl, 2), "L/h;  dose ≈", round(400 * cl, -1), "mg/day")

**Teaching data only.** Drug V and these numbers are fictional. Real dosing follows the label, local protocols and TDM.